In [7]:
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
import joblib

In [8]:
data = pd.read_csv("../dataset/dataset.csv", dtype={'teamId' : str})
data_elo = pd.read_csv("../dataset/nba_elo.csv", dtype={'teamId' : str})
data

,teamId,gameId,GAME_DATE,DUEL,WL,NB_VICTOIRE,NB_DEFAITE,PCT_VICTOIRE,DUREE_MATCH,TIR REUSSI,...,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,1610612748,21300002,2013-10-29,MIA vs. CHI,W,1,0,1.000,240,37,...,20.2,0.590,0.631,1.0,0.197,100.44,98.5,82.08,99.0,0.599
1,1610612747,21300003,2013-10-29,LAL vs. LAC,W,1,0,1.000,240,42,...,19.2,0.527,0.551,1.0,0.195,102.72,98.5,82.08,99.0,0.503
2,1610612746,21300003,2013-10-29,LAC @ LAL,L,0,1,0.000,240,41,...,16.3,0.542,0.553,1.0,0.200,102.72,98.5,82.08,98.0,0.497
3,1610612754,21300001,2013-10-29,IND vs. ORL,W,1,0,1.000,240,34,...,22.3,0.528,0.570,1.0,0.198,99.74,94.0,78.33,94.0,0.661
4,1610612741,21300002,2013-10-29,CHI @ MIA,L,0,1,0.000,240,35,...,19.4,0.464,0.510,1.0,0.198,100.44,98.5,82.08,98.0,0.401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24077,1610612739,22301187,2024-04-14,CLE vs. CHA,L,48,34,0.585,240,44,...,14.4,0.559,0.567,1.0,0.196,97.40,97.0,80.83,97.0,0.427
24078,1610612761,22301189,2024-04-14,TOR @ MIA,L,25,57,0.305,240,38,...,19.2,0.472,0.519,1.0,0.198,103.30,100.0,83.33,99.0,0.369
24079,1610612742,22301196,2024-04-14,DAL @ OKC,L,50,32,0.610,240,32,...,14.3,0.371,0.412,1.0,0.199,107.76,105.5,87.92,105.0,0.246
24080,1610612754,22301188,2024-04-14,IND vs. ATL,W,47,35,0.573,240,65,...,15.0,0.745,0.752,1.0,0.193,108.64,107.0,89.17,107.0,0.649


In [9]:
def get_last_10_perf(team, data):
    nb_games="10"
    df_last_10_games = data[data['teamTricode'] == team].tail(10)
    #df_last_10_games
    df_stat_last_perf = pd.DataFrame(columns=['NB_WIN_L10', 'PCT_TIR_REUSSI_L10','PCT_3PTS_L10', 
                               'PCT_LANCER_FRANC_L10','estimatedOffensiveRating_L10', 'offensiveRating_L10',
                               'estimatedDefensiveRating_L10',
                               'defensiveRating_L10','estimatedNetRating_L10', 'netRating_L10', 'assistPercentage_L10',
                               'assistToTurnover_L10', 'assistRatio_L10','estimatedTeamTurnoverPercentage_L10', 'turnoverRatio_L10',
                               'effectiveFieldGoalPercentage_L10','trueShootingPercentage_L10', 'estimatedPace_L10', 'pace_L10',
                               'pacePer40_L10', 'PIE_L10'])
    df_last_10_games.reset_index(drop=True, inplace=True)
    new_row = {"NB_WIN_L"+nb_games : (df_last_10_games['WL'] == 'W').sum() / int(nb_games),
                       "PCT_TIR_REUSSI_L"+nb_games : df_last_10_games['PCT_TIR_REUSSI'].mean(),
                       "PCT_3PTS_L"+nb_games : df_last_10_games['PCT_3PTS'].mean(),
                       "PCT_LANCER_FRANC_L"+nb_games : df_last_10_games['PCT_LANCER_FRANC'].mean(),
                       "estimatedOffensiveRating_L"+nb_games : df_last_10_games['estimatedOffensiveRating'].mean(),
                       "offensiveRating_L"+nb_games : df_last_10_games['offensiveRating'].mean(),
                       "estimatedDefensiveRating_L"+nb_games : df_last_10_games['estimatedDefensiveRating'].mean(),
                       "defensiveRating_L"+nb_games : df_last_10_games['defensiveRating'].mean(),
                       "estimatedNetRating_L"+nb_games : df_last_10_games['estimatedNetRating'].mean(),
                       "netRating_L"+nb_games : df_last_10_games['netRating'].mean(),
                       "assistPercentage_L"+nb_games : df_last_10_games['assistPercentage'].mean(),
                       "assistToTurnover_L"+nb_games : df_last_10_games['assistToTurnover'].mean(),
                       "assistRatio_L"+nb_games : df_last_10_games['assistRatio'].mean(),
                       "estimatedTeamTurnoverPercentage_L"+nb_games : df_last_10_games['estimatedTeamTurnoverPercentage'].mean(),
                       "turnoverRatio_L"+nb_games : df_last_10_games['turnoverRatio'].mean(),
                       "effectiveFieldGoalPercentage_L"+nb_games : df_last_10_games['effectiveFieldGoalPercentage'].mean(),
                       "trueShootingPercentage_L"+nb_games : df_last_10_games['trueShootingPercentage'].mean(),
                       "estimatedPace_L"+nb_games : df_last_10_games['estimatedPace'].mean(),
                       "pace_L"+nb_games : df_last_10_games['pace'].mean(),
                       "pacePer40_L"+nb_games : df_last_10_games['pacePer40'].mean(),
                       "PIE_L"+nb_games : df_last_10_games['PIE'].mean()}
    df_stat_last_perf.loc[len(df_stat_last_perf)] = new_row
    return df_stat_last_perf

In [10]:
home_team = "LAC"
away_team = "SAS"
def get_elo(team_name, data_elo):
    df_elo_home = data_elo[data_elo['team1'] == team_name]
    df_elo_away = data_elo[data_elo['team2'] == team_name]
    
    if datetime.strptime(df_elo_home['date'].iloc[-1], '%Y-%m-%d') > datetime.strptime(df_elo_away['date'].iloc[-1], '%Y-%m-%d'):
        return df_elo_home['elo1_post'].iloc[-1]
    else :
        return df_elo_away['elo2_post'].iloc[-1]

stat_home = get_last_10_perf(home_team, data)
stat_home['ELO'] = get_elo(home_team, data_elo)
stat_away = get_last_10_perf(away_team, data)
stat_away['ELO'] = get_elo(away_team, data_elo)
stat_away

,NB_WIN_L10,PCT_TIR_REUSSI_L10,PCT_3PTS_L10,PCT_LANCER_FRANC_L10,estimatedOffensiveRating_L10,offensiveRating_L10,estimatedDefensiveRating_L10,defensiveRating_L10,estimatedNetRating_L10,netRating_L10,...,assistRatio_L10,estimatedTeamTurnoverPercentage_L10,turnoverRatio_L10,effectiveFieldGoalPercentage_L10,trueShootingPercentage_L10,estimatedPace_L10,pace_L10,pacePer40_L10,PIE_L10,ELO
0,0.6,0.4736,0.368,0.7694,108.91,109.49,106.12,108.66,2.79,0.84,...,21.76,16.07,16.14,0.5544,0.5807,102.804,101.292,84.411,0.5174,1377.219751


In [11]:
stat_home

,NB_WIN_L10,PCT_TIR_REUSSI_L10,PCT_3PTS_L10,PCT_LANCER_FRANC_L10,estimatedOffensiveRating_L10,offensiveRating_L10,estimatedDefensiveRating_L10,defensiveRating_L10,estimatedNetRating_L10,netRating_L10,...,assistRatio_L10,estimatedTeamTurnoverPercentage_L10,turnoverRatio_L10,effectiveFieldGoalPercentage_L10,trueShootingPercentage_L10,estimatedPace_L10,pace_L10,pacePer40_L10,PIE_L10,ELO
0,0.6,0.4665,0.3473,0.8629,110.33,111.47,106.88,109.2,3.45,2.25,...,17.81,13.9932,14.13,0.5295,0.5713,101.046,99.45,82.875,0.5058,1545.758104


In [12]:
stat = []
for column in stat_home.columns :
    stat.append(stat_home[column].iloc[-1] - stat_away[column].iloc[-1])
stat

[0.0,
 -0.007100000000000051,
 -0.02069999999999994,
 0.09350000000000014,
 1.4200000000000017,
 1.9799999999999898,
 0.7600000000000193,
 0.5400000000000063,
 0.6600000000000001,
 1.4100000000000001,
 -0.16789999999999994,
 -0.18699999999999983,
 -3.9499999999999993,
 -2.076800000000002,
 -2.0100000000000016,
 -0.024900000000000033,
 -0.009399999999999853,
 -1.7579999999999956,
 -1.8419999999999987,
 -1.5360000000000156,
 -0.011599999999999944,
 168.53835299999992]

In [13]:
stat_home.columns.tolist()

['NB_WIN_L10',
 'PCT_TIR_REUSSI_L10',
 'PCT_3PTS_L10',
 'PCT_LANCER_FRANC_L10',
 'estimatedOffensiveRating_L10',
 'offensiveRating_L10',
 'estimatedDefensiveRating_L10',
 'defensiveRating_L10',
 'estimatedNetRating_L10',
 'netRating_L10',
 'assistPercentage_L10',
 'assistToTurnover_L10',
 'assistRatio_L10',
 'estimatedTeamTurnoverPercentage_L10',
 'turnoverRatio_L10',
 'effectiveFieldGoalPercentage_L10',
 'trueShootingPercentage_L10',
 'estimatedPace_L10',
 'pace_L10',
 'pacePer40_L10',
 'PIE_L10',
 'ELO']

In [14]:
stat

[0.0,
 -0.007100000000000051,
 -0.02069999999999994,
 0.09350000000000014,
 1.4200000000000017,
 1.9799999999999898,
 0.7600000000000193,
 0.5400000000000063,
 0.6600000000000001,
 1.4100000000000001,
 -0.16789999999999994,
 -0.18699999999999983,
 -3.9499999999999993,
 -2.076800000000002,
 -2.0100000000000016,
 -0.024900000000000033,
 -0.009399999999999853,
 -1.7579999999999956,
 -1.8419999999999987,
 -1.5360000000000156,
 -0.011599999999999944,
 168.53835299999992]

In [15]:
data = {colonne: [valeur] for colonne, valeur in zip(stat_home.columns.tolist(), stat)}
df = pd.DataFrame(data)
df

,NB_WIN_L10,PCT_TIR_REUSSI_L10,PCT_3PTS_L10,PCT_LANCER_FRANC_L10,estimatedOffensiveRating_L10,offensiveRating_L10,estimatedDefensiveRating_L10,defensiveRating_L10,estimatedNetRating_L10,netRating_L10,...,assistRatio_L10,estimatedTeamTurnoverPercentage_L10,turnoverRatio_L10,effectiveFieldGoalPercentage_L10,trueShootingPercentage_L10,estimatedPace_L10,pace_L10,pacePer40_L10,PIE_L10,ELO
0,0.0,-0.0071,-0.0207,0.0935,1.42,1.98,0.76,0.54,0.66,1.41,...,-3.95,-2.0768,-2.01,-0.0249,-0.0094,-1.758,-1.842,-1.536,-0.0116,168.538353


In [16]:
df.columns

Index(['NB_WIN_L10', 'PCT_TIR_REUSSI_L10', 'PCT_3PTS_L10',
       'PCT_LANCER_FRANC_L10', 'estimatedOffensiveRating_L10',
       'offensiveRating_L10', 'estimatedDefensiveRating_L10',
       'defensiveRating_L10', 'estimatedNetRating_L10', 'netRating_L10',
       'assistPercentage_L10', 'assistToTurnover_L10', 'assistRatio_L10',
       'estimatedTeamTurnoverPercentage_L10', 'turnoverRatio_L10',
       'effectiveFieldGoalPercentage_L10', 'trueShootingPercentage_L10',
       'estimatedPace_L10', 'pace_L10', 'pacePer40_L10', 'PIE_L10', 'ELO'],
      dtype='object')

In [17]:

df.drop(columns=['NB_WIN_L10','PCT_TIR_REUSSI_L10', 'PCT_3PTS_L10', 'effectiveFieldGoalPercentage_L10','trueShootingPercentage_L10'])

new_order = ['ELO', 'PCT_LANCER_FRANC_L10', 'PIE_L10', 'assistPercentage_L10',
       'assistRatio_L10', 'assistToTurnover_L10', 'defensiveRating_L10',
       'estimatedDefensiveRating_L10', 'estimatedNetRating_L10',
       'estimatedOffensiveRating_L10', 'estimatedPace_L10',
       'estimatedTeamTurnoverPercentage_L10', 'netRating_L10',
       'offensiveRating_L10', 'pacePer40_L10', 'pace_L10',
       'turnoverRatio_L10']
df = df.reindex(columns=new_order)
df

,ELO,PCT_LANCER_FRANC_L10,PIE_L10,assistPercentage_L10,assistRatio_L10,assistToTurnover_L10,defensiveRating_L10,estimatedDefensiveRating_L10,estimatedNetRating_L10,estimatedOffensiveRating_L10,estimatedPace_L10,estimatedTeamTurnoverPercentage_L10,netRating_L10,offensiveRating_L10,pacePer40_L10,pace_L10,turnoverRatio_L10
0,168.538353,0.0935,-0.0116,-0.1679,-3.95,-0.187,0.54,0.76,0.66,1.42,-1.758,-2.0768,1.41,1.98,-1.536,-1.842,-2.01


In [20]:
import json

# Chemin vers le fichier téléchargé
file_path = '../dataset/team_name.json'

# Charger le contenu du fichier JSON dans un dictionnaire
with open(file_path, 'r') as file:
    team_dict = json.load(file)

# Afficher le dictionnaire pour vérification
team_dict


{'BOS': 'Boston Celtics',
 'NYK': 'New York Knicks',
 'ATL': 'Atlanta Hawks',
 'BKN': 'Brooklyn Nets',
 'CHA': 'Charlotte Hornets',
 'CHI': 'Chicago Bulls',
 'CLE': 'Cleveland Cavaliers',
 'DEN': 'Denver Nuggets',
 'DAL': 'Dallas Mavericks',
 'DET': 'Detroit Pistons',
 'GSW': 'Golden State Warriors',
 'HOU': 'Houston Rockets',
 'IND': 'Indiana Pacers',
 'LAC': 'Los Angeles Clippers',
 'LAL': 'Los Angeles Lakers',
 'MEM': 'Memphis Grizzlies',
 'MIA': 'Miami Heat',
 'MIL': 'Milwaukee Bucks',
 'MIN': 'Minnesota Timberwolves',
 'NOP': 'New Orleans Pelicans',
 'OKC': 'Oklahoma City Thunder',
 'ORL': 'Orlando Magic',
 'PHI': 'Philadelphia 76ers',
 'PHX': 'Phoenix Suns',
 'POR': 'Portland Trail Blazers',
 'SAC': 'Sacramento Kings',
 'SAS': 'San Antonio Spurs',
 'TOR': 'Toronto Raptors',
 'UTA': 'Utah Jazz',
 'WAS': 'Washington Wizards'}

In [27]:
import joblib
import json
import pandas as pd



def get_prediction(home_team, away_team):
    with open('../dataset/team_name.json', 'r') as file:
        team_dict = json.load(file)
        
    data = pd.read_csv("../dataset/dataset.csv", dtype={'teamId' : str})
    data_elo = pd.read_csv("../dataset/nba_elo.csv", dtype={'teamId' : str})
    data_elo['team1'] = data_elo['team1'].replace({'PHO': 'PHX', 'BRK': 'BKN', 'CHO': 'CHA'})
    data_elo['team2'] = data_elo['team2'].replace({'PHO': 'PHX', 'BRK': 'BKN', 'CHO': 'CHA'})
    data_elo['team1'] = data_elo['team1'].replace(team_dict)
    data_elo['team2'] = data_elo['team2'].replace(team_dict)
    data['teamTricode'] = data['teamTricode'].replace(team_dict)
    # Charger le contenu du fichier JSON dans un dictionnaire
    
        
    stat_home = get_last_10_perf(home_team, data)
    stat_home['ELO'] = get_elo(home_team, data_elo)
    stat_away = get_last_10_perf(away_team, data)
    stat_away['ELO'] = get_elo(away_team, data_elo)
    
    stat = []
    for column in stat_home.columns :
        stat.append(stat_home[column].iloc[-1] - stat_away[column].iloc[-1])
    data = {colonne: [valeur] for colonne, valeur in zip(stat_home.columns.tolist(), stat)}
    df = pd.DataFrame(data)
    df.drop(columns=['NB_WIN_L10','PCT_TIR_REUSSI_L10', 'PCT_3PTS_L10', 'effectiveFieldGoalPercentage_L10','trueShootingPercentage_L10'])

    new_order = ['ELO', 'PCT_LANCER_FRANC_L10', 'PIE_L10', 'assistPercentage_L10',
       'assistRatio_L10', 'assistToTurnover_L10', 'defensiveRating_L10',
       'estimatedDefensiveRating_L10', 'estimatedNetRating_L10',
       'estimatedOffensiveRating_L10', 'estimatedPace_L10',
       'estimatedTeamTurnoverPercentage_L10', 'netRating_L10',
       'offensiveRating_L10', 'pacePer40_L10', 'pace_L10',
       'turnoverRatio_L10']
    df = df.reindex(columns=new_order)
    
    # Charger le modèle et le scaler sauvegardés
    model = joblib.load('KNN.pkl')
    scaler = joblib.load('scaler.pkl')
    
    # Supposons que X_new sont vos nouvelles données
    #X_new = ...  # nouvelles données
    
    # Transformer les nouvelles données avec le scaler chargé
    X_new_scaled = scaler.transform(df)
    
    # Faire des prédictions avec le modèle chargé
    predictions = model.predict(X_new_scaled)
    if predictions[0] == 1:
        return home_team
    else:
        return away_team
    

get_prediction("Los Angeles Lakers", "San Antonio Spurs")

'Los Angeles Lakers'